In [3]:
import time
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import keras_tuner as kt


In [4]:
# Setup: Load Dataset
# ==========================================
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define parameter grid with exactly 36 total combinations (3 x 4 x 3 = 36)
param_grid = {
    'max_depth': [3, 5, 10],            # 3 values
    'min_samples_split': [2, 5, 10, 15], # 4 values
    'criterion': ['gini', 'entropy', 'log_loss'] # 3 values
}

In [5]:
# Task 1: GridSearchCV on Decision Tree
# ==========================================
print("--- Task 1: Running GridSearchCV ---")
dt_clf = DecisionTreeClassifier(random_state=42)

start_time = time.time()
grid_search = GridSearchCV(
    estimator=dt_clf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    return_train_score=False
)
grid_search.fit(X_train, y_train)
grid_time = time.time() - start_time
print(f"GridSearch completed in {grid_time:.4f} seconds.")

--- Task 1: Running GridSearchCV ---
GridSearch completed in 8.4897 seconds.


In [6]:
# Task 2: Compare all 36 combinations & Identify Top 3
# ==========================================
print("\n--- Task 2: Top 3 Parameter Combinations (GridSearch) ---")
grid_results = pd.DataFrame(grid_search.cv_results_)
top_3_grid = grid_results.sort_values(by='mean_test_score', ascending=False).head(3)
print(top_3_grid[['params', 'mean_test_score', 'rank_test_score']])



--- Task 2: Top 3 Parameter Combinations (GridSearch) ---
                                               params  mean_test_score  \
18  {'criterion': 'entropy', 'max_depth': 5, 'min_...         0.947253   
34  {'criterion': 'log_loss', 'max_depth': 10, 'mi...         0.947253   
30  {'criterion': 'log_loss', 'max_depth': 5, 'min...         0.947253   

    rank_test_score  
18                1  
34                1  
30                1  


In [7]:
# Task 3: RandomizedSearchCV with 30 iterations
# ==========================================
print("\n--- Task 3: Running RandomizedSearchCV (30 iterations) ---")
start_time = time.time()
random_search = RandomizedSearchCV(
    estimator=dt_clf,
    param_distributions=param_grid,
    n_iter=30,
    cv=5,
    scoring='accuracy',
    random_state=42
)
random_search.fit(X_train, y_train)
random_time = time.time() - start_time
print(f"RandomizedSearch completed in {random_time:.4f} seconds.")



--- Task 3: Running RandomizedSearchCV (30 iterations) ---
RandomizedSearch completed in 7.1079 seconds.


In [8]:
# Task 4: Compare Grid vs Random Search
# ==========================================
print("\n--- Task 4: Grid vs Random Search Comparison ---")
comparison_df = pd.DataFrame({
    'Search Method': ['GridSearchCV', 'RandomizedSearchCV'],
    'Total Combinations Sampled': [len(grid_results), 30],
    'Execution Time (s)': [grid_time, random_time],
    'Best Cross-Val Accuracy': [grid_search.best_score_, random_search.best_score_],
    'Best Parameters': [grid_search.best_params_, random_search.best_params_]
})
print(comparison_df.to_string(index=False))


--- Task 4: Grid vs Random Search Comparison ---
     Search Method  Total Combinations Sampled  Execution Time (s)  Best Cross-Val Accuracy                                                    Best Parameters
      GridSearchCV                          36            8.489697                 0.947253  {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 10}
RandomizedSearchCV                          30            7.107935                 0.947253 {'min_samples_split': 10, 'max_depth': 5, 'criterion': 'log_loss'}


In [2]:
!pip install keras-tuner

Defaulting to user installation because normal site-packages is not writeable

   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   ---------------------------------------- 0/2 [kt-legacy]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
   -------------------- ------------------- 1/2 [keras-tuner]
 


[notice] A new release of pip is available: 26.1.2 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
# ==========================================
# Task 5: Keras Tuner for Neural Networks
# Tune Learning Rate and Batch Size
# ==========================================

import os
from datetime import datetime

import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

print("\n--- Task 5: Tuning Neural Network with Keras Tuner ---")


# --------------------------------------------------
# 1. Create a safe results directory
# --------------------------------------------------

tuner_directory = r"C:\keras_tuner_results"

os.makedirs(tuner_directory, exist_ok=True)

# Create a unique project folder.
# This prevents Windows from trying to delete a locked old folder.
project_name = "nn_tuning_" + datetime.now().strftime("%Y%m%d_%H%M%S")


# --------------------------------------------------
# 2. Define the HyperModel
# --------------------------------------------------

class NeuralNetworkHyperModel(kt.HyperModel):

    def build(self, hp):

        # Tune the learning rate
        learning_rate = hp.Choice(
            "learning_rate",
            values=[1e-2, 1e-3, 1e-4]
        )

        model = keras.Sequential([
            keras.Input(shape=(X_train.shape[1],)),

            layers.Dense(
                32,
                activation="relu"
            ),

            layers.Dense(
                1,
                activation="sigmoid"
            )
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model


    def fit(self, hp, model, x, y, **kwargs):

        # Tune the batch size
        batch_size = hp.Choice(
            "batch_size",
            values=[16, 32]
        )

        return model.fit(
            x,
            y,
            batch_size=batch_size,
            **kwargs
        )


# --------------------------------------------------
# 3. Create the Random Search tuner
# --------------------------------------------------

tuner = kt.RandomSearch(
    hypermodel=NeuralNetworkHyperModel(),
    objective="val_accuracy",
    max_trials=6,
    executions_per_trial=1,
    directory=tuner_directory,
    project_name=project_name,

    # False prevents Keras Tuner from deleting old folders
    overwrite=False
)


# --------------------------------------------------
# 4. Display the search space
# --------------------------------------------------

print("\nHyperparameter Search Space:")
tuner.search_space_summary()


# --------------------------------------------------
# 5. Start hyperparameter tuning
# --------------------------------------------------

tuner.search(
    X_train,
    y_train,
    epochs=20,
    validation_split=0.2,
    verbose=1
)


# --------------------------------------------------
# 6. Get the best hyperparameters
# --------------------------------------------------

best_nn_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print("\n--- Best Hyperparameters ---")

print(
    "Optimal Learning Rate:",
    best_nn_hps.get("learning_rate")
)

print(
    "Optimal Batch Size:",
    best_nn_hps.get("batch_size")
)


# --------------------------------------------------
# 7. Get and evaluate the best model
# --------------------------------------------------

best_nn_model = tuner.get_best_models(
    num_models=1
)[0]

test_loss, test_accuracy = best_nn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n--- Best Model Performance ---")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Trial 6 Complete [00h 00m 05s]
val_accuracy: 0.8241758346557617

Best val_accuracy So Far: 0.9340659379959106
Total elapsed time: 00h 00m 41s

--- Best Hyperparameters ---
Optimal Learning Rate: 0.01
Optimal Batch Size: 16

--- Best Model Performance ---
Test Loss:     0.1513
Test Accuracy: 0.9474


In [10]:
# ==========================================
# Task 5: Keras Tuner for Neural Networks
# Tune Learning Rate and Batch Size
# ==========================================

import os
from datetime import datetime

import keras_tuner as kt
from tensorflow import keras
from tensorflow.keras import layers

print("\n--- Task 5: Tuning Neural Network with Keras Tuner ---")


# --------------------------------------------------
# 1. Create a safe results directory
# --------------------------------------------------

tuner_directory = r"C:\keras_tuner_results"

os.makedirs(tuner_directory, exist_ok=True)

# Create a unique project folder.
# This prevents Windows from trying to delete a locked old folder.
project_name = "nn_tuning_" + datetime.now().strftime("%Y%m%d_%H%M%S")


# --------------------------------------------------
# 2. Define the HyperModel
# --------------------------------------------------

class NeuralNetworkHyperModel(kt.HyperModel):

    def build(self, hp):

        # Tune the learning rate
        learning_rate = hp.Choice(
            "learning_rate",
            values=[1e-2, 1e-3, 1e-4]
        )

        model = keras.Sequential([
            keras.Input(shape=(X_train.shape[1],)),

            layers.Dense(
                32,
                activation="relu"
            ),

            layers.Dense(
                1,
                activation="sigmoid"
            )
        ])

        model.compile(
            optimizer=keras.optimizers.Adam(
                learning_rate=learning_rate
            ),
            loss="binary_crossentropy",
            metrics=["accuracy"]
        )

        return model


    def fit(self, hp, model, x, y, **kwargs):

        # Tune the batch size
        batch_size = hp.Choice(
            "batch_size",
            values=[16, 32]
        )

        return model.fit(
            x,
            y,
            batch_size=batch_size,
            **kwargs
        )


# --------------------------------------------------
# 3. Create the Random Search tuner
# --------------------------------------------------

tuner = kt.RandomSearch(
    hypermodel=NeuralNetworkHyperModel(),
    objective="val_accuracy",
    max_trials=6,
    executions_per_trial=1,
    directory=tuner_directory,
    project_name=project_name,

    # False prevents Keras Tuner from deleting old folders
    overwrite=False
)


# --------------------------------------------------
# 4. Display the search space
# --------------------------------------------------

print("\nHyperparameter Search Space:")
tuner.search_space_summary()


# --------------------------------------------------
# 5. Start hyperparameter tuning
# --------------------------------------------------

tuner.search(
    X_train,
    y_train,
    epochs=20,
    validation_split=0.2,
    verbose=1
)


# --------------------------------------------------
# 6. Get the best hyperparameters
# --------------------------------------------------

best_nn_hps = tuner.get_best_hyperparameters(
    num_trials=1
)[0]

print("\n--- Best Hyperparameters ---")

print(
    "Optimal Learning Rate:",
    best_nn_hps.get("learning_rate")
)

print(
    "Optimal Batch Size:",
    best_nn_hps.get("batch_size")
)


# --------------------------------------------------
# 7. Get and evaluate the best model
# --------------------------------------------------

best_nn_model = tuner.get_best_models(
    num_models=1
)[0]

test_loss, test_accuracy = best_nn_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print("\n--- Best Model Performance ---")
print(f"Test Loss:     {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

Trial 6 Complete [00h 00m 05s]
val_accuracy: 0.9230769276618958

Best val_accuracy So Far: 0.9230769276618958
Total elapsed time: 00h 00m 31s

--- Best Hyperparameters ---
Optimal Learning Rate: 0.01
Optimal Batch Size: 32

--- Best Model Performance ---
Test Loss:     0.2021
Test Accuracy: 0.9386


In [11]:
# Task 6: Final Model Documentation & Evaluation
# ==========================================
print("\n--- Task 6: Final Model Documentation & Evaluation ---")

# Best Decision Tree Test Evaluation
best_dt_model = grid_search.best_estimator_
dt_test_acc = best_dt_model.score(X_test, y_test)

# Train Best Neural Network with optimal parameters
best_nn_model = tuner.hypermodel.build(best_nn_hps)
best_nn_model.fit(
    X_train, y_train,
    epochs=30,
    batch_size=best_nn_hps.get('batch_size'),
    verbose=0
)
_, nn_test_acc = best_nn_model.evaluate(X_test, y_test, verbose=0)

summary_table = pd.DataFrame({
    'Model Type': ['Decision Tree (Best Grid)', 'Neural Network (Best Keras Tuner)'],
    'Optimal Parameters': [
        f"{grid_search.best_params_}", 
        f"lr={best_nn_hps.get('learning_rate')}, batch_size={best_nn_hps.get('batch_size')}"
    ],
    'Test Accuracy': [dt_test_acc, nn_test_acc]
})

print("\n=== FINAL HYPERPARAMETER TUNING SUMMARY ===")
print(summary_table.to_string(index=False))


--- Task 6: Final Model Documentation & Evaluation ---

=== FINAL HYPERPARAMETER TUNING SUMMARY ===
                       Model Type                                                Optimal Parameters  Test Accuracy
        Decision Tree (Best Grid) {'criterion': 'entropy', 'max_depth': 5, 'min_samples_split': 10}       0.956140
Neural Network (Best Keras Tuner)                                            lr=0.01, batch_size=32       0.964912
